In [1]:
import newkernelo as lib
import numpy as np
import time
import os.path
import json
import logging
logging.getLogger().setLevel(logging.INFO)

In [2]:
# model =lib.FunctionalModel()
# help(lib.FunctionalModel)
# a = model.get_D_dimension()
# print(model)

In [3]:
model = lib.TestModel()
print(model.__doc__)
print(lib.ShkuratovModel.__doc__)
# print(lib.FunctionalModel.__doc__)
# print(lib.ExternalPythonModel.__doc__)
# print(lib.ExternalPythonModel.__init__.__doc__)
help(lib.TestModel)




            The class :class:`TestModel` is low-dimensional functional model implemented in order to test the GLLiM method with simple but not trivial example.
            The functional F is designed so as to exhibit 2 solutions with D=9 and L=4. :math:`F = A ◦ G ◦ H`, where
             - :math:`A` is a (DxL) injective matrix,
            
                .. math::
                    A = \frac{1}{2} \begin{pmatrix}
                        1 & 2 & 2 & 1 \\
                        0 & 0.5 & 0 & 0 \\
                        0 & 0 & 1 & 0 \\
                        0 & 0 & 0 & 3 \\
                        0.2 & 0 & 0 & 0 \\
                        0 & -0.5 & 0 & 0 \\
                        -0.2 & 0 & -1 & 0 \\
                        -1 & 0 & 2 & 0 \\
                        0 & 0 & 0 & -0.7
                    \end{pmatrix}

             - :math:`G(x)` = :math:`(exp(x_1), exp(x_2), exp(x_3), exp(x_4))` and 
             - :math:`H(x)` = :math:`(x_1, x_2, 4(x_3-0.5)^2, x_4)`.
       

## Test Model

In [4]:
model = lib.TestModel()
e = np.exp(1)
x_test = np.zeros(4)
y_test = model.F(x_test)
true_result = np.array([4+2*e, 0.5, e, 3, 0.2, -0.5, -0.2-e, 2*e-1, -0.7])*0.5
print("y_test")
print(y_test)
print("true_result")
print(true_result)

y_test
[ 4.71828183  0.25        1.35914091  1.5         0.1        -0.25
 -1.45914091  2.21828183 -0.35      ]
true_result
[ 4.71828183  0.25        1.35914091  1.5         0.1        -0.25
 -1.45914091  2.21828183 -0.35      ]


In [5]:
e = np.exp(1)
x = np.ones(4)

# y = np.zeros(9)
# t0 = time.time()
# for i in range(10000000): #10000000
#     model.F(x,y)
# tf = time.time() - t0
# print("Time F by reference: {}".format(tf))

# t0 = time.time()
# for i in range(10000000):
#     z = model.F(x)
# tf = time.time() - t0
# print("Time F by value: {}".format(tf))

# true_result = np.array([4+2*e, 0.5, e, 3, 0.2, -0.5, -0.2-e, 2*e-1, -0.7])*0.5
# print("true_result")
# print(true_result)

z = model.F(x)
print(z)
print(x)
print(x.shape)
model.to_physic(x) # does nothing
print(x)
print(x.shape)
w = model.to_physic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
print(x)
model.from_physic(x)
print(x)
y = model.from_physic(x)
print(x)
print(y)
z = model.from_physic(y)
print(x)
print(y)
print(z)

[ 8.15484549  0.67957046  1.35914091  4.07742274  0.27182818 -0.67957046
 -1.6309691   1.35914091 -0.95139864]
[1. 1. 1. 1.]
(4,)
[1. 1. 1. 1.]
(4,)
[1. 1. 1. 1.]
[2. 2. 2. 2.]
(4,)
=========== From physic ==========
[1. 1. 1. 1.]
[1. 1. 1. 1.]
[1. 1. 1. 1.]
[0.5 0.5 0.5 0.5]
[1. 1. 1. 1.]
[0.5 0.5 0.5 0.5]
[0.25 0.25 0.25 0.25]


## Shkuratov

In [6]:
print(os.getcwd())
with open('../../pytest/new_test_shkuratov.json', 'r') as f:
    data = json.load(f)

D = 50
row_size = D
col_size = 3

scalingCoeffs = [1.0,1.5,1.5,1.5,1.5]
offset = [0,0,0.2,0,0]


# # Create JSON file with geometries
geometries = np.empty((row_size,col_size))
var_geom = ["inc", "eme", "phi"]
for j in range(3):
    i=0
    for v in data[var_geom[j]]:
        geometries[i,j] = v # geometries.shape = (D,3)
        i+=1
# print(geometries)
print(geometries.shape)

# geom = {'geometries': [[geometries[j,:].tolist()] for j in range(3)]
# }
# with open('geometries_shkuratov.json', 'w') as fp:
#     json.dump(geom, fp)

## INTEGRATION au code C++
variant = "5p"
physicalModel = lib.ShkuratovModel(geometries, variant, scalingCoeffs, offset)


### TEST
N = 100
L = 5
variables = ["an", "mu1", "nu", "m", "mu2"]
photometries = np.empty((L,N))

# Read photometries
for l in range(L):
    for n in range(N):
        photometries[l,n] = (float(data[variables[l]][n]) - offset[l]) / scalingCoeffs[l]
        n+=1


# Read expected results
expected_results = np.empty((D,N))
n=0
for n in range(N):
    for d in range(D):
        expected_results[d,n] = float(data["y"][n][d])



# compute results from the model
# result = np.empty((D,))
assert_list = []
for n in range(N):
    result = physicalModel.F(photometries[:,n])
    assert_list.append(np.allclose(expected_results[:,n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[:10,n])
print(result[:10])


/home/luc/Documents/dev/kernelo-gllim-is/new_kernelo/python
(50, 3)
(50,)
[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
False
True
[0.01652884 0.00787757 0.023348   0.01033272 0.00816459 0.02172611
 0.0126815  0.04004887 0.01367903 0.02221388]
[0.01652884 0.00787757 0.023348   0.01033272 0.00816459 0.02172611
 0.0126815  0.04004887 0.01367903 0.02221388]


## Hapke

In [7]:
with open('../../data_ref_hapke6p_geom70.json') as json_file:
    data = json.load(json_file)
    expected_results = np.array(data["data_ref"]["synthetic_dataset"]['Y'])
    geometries = np.array(data["data_ref"]['geometries'])
    photometries = np.array(data["data_ref"]["synthetic_dataset"]['X'])
print(geometries.shape)
print(photometries.shape)
print(expected_results.shape)


hapkeModel = lib.HapkeModel(geometries, "2002", "six", 30.0,1,0)
print(hapkeModel.get_D_dimension())
print(hapkeModel.get_L_dimension())

# TEST
N = 10
assert_list = []
for n in range(N):
    result = hapkeModel.F(photometries[n])
    assert_list.append(np.allclose(expected_results[n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[n])
print(result)

(70, 3)
(20000, 6)
(20000, 70)
70
6
(70,)
[True, True, True, True, True, True, True, True, True, True]
False
True
[0.661713   0.71376229 0.75547619 0.79594483 0.84898809 0.94010314
 1.09388016 1.09388016 0.94010314 0.84898809 0.79594483 0.75547619
 0.71376229 0.661713   0.67201986 0.71167323 0.74084312 0.76380446
 0.78506504 0.81104192 0.85408973 0.94010314 1.09745333 1.10620349
 0.93548503 0.82486549 0.74998315 0.68140027 0.79251966 0.94548664
 1.15975826 1.12410881 0.93548503 0.83902679 0.79594483 0.77532515
 0.76380446 0.75567048 0.74812848 0.73961698 0.72954369 0.71933715
 0.89568301 0.79645035 0.75188174 0.72954369 0.71763    0.71167323
 0.71025644 0.71376229 0.72469036 0.74998315 0.80807864 0.94548664
 1.23267475 1.39492325 0.67631355 0.73944407 0.8019977  0.88319617
 0.99919454 1.055518   0.94010314 0.85957158 0.81506496 0.78747365
 0.7650571  0.74117638 0.71104034 0.67001222]
[0.661713   0.71376229 0.75547619 0.79594483 0.84898809 0.94010314
 1.09388016 1.09388016 0.94010314 0.

In [8]:
with open('../../data_ref_hapke4p_geom70.json') as json_file:
    data = json.load(json_file)
    expected_results = np.array(data["data_ref"]["synthetic_dataset"]['Y'])
    geometries = np.array(data["data_ref"]['geometries'])
    photometries = np.array(data["data_ref"]["synthetic_dataset"]['X'])
print(geometries.shape)
print(photometries.shape)
print(expected_results.shape)


hapkeModel = lib.HapkeModel(geometries, "2002", "four", 30.0,1,0)
print(hapkeModel.get_D_dimension())
print(hapkeModel.get_L_dimension())

# TEST
N = 10
assert_list = []
for n in range(N):
    result = hapkeModel.F(photometries[n])
    assert_list.append(np.allclose(expected_results[n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[n,:10])
print(result[:10])

### NOTE : le expected_results[n][2] et result[2] sont différents.... mais les autres sont égaux. à voir...

(70, 3)
(20000, 4)
(20000, 70)
70
4
(70,)
[True, True, True, True, True, True, True, True, True, True]
False
True
[0.65998213 0.71192246 0.75322617 0.79272546 0.84355547 0.92952621
 1.07279062 1.07279062 0.92952621 0.84355547]
[0.65998213 0.71192246 0.75322617 0.79272546 0.84355547 0.92952621
 1.07279062 1.07279062 0.92952621 0.84355547]


In [9]:
print(hapkeModel.get_D_dimension())
print(hapkeModel.get_L_dimension())
x = np.ones(hapkeModel.get_L_dimension()) / 10

print(x)
print(x.shape)
hapkeModel.to_physic(x) # does nothing
print(x)
print(x.shape)
w = hapkeModel.to_physic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
print(x)
hapkeModel.from_physic(x)
print(x)
y = hapkeModel.from_physic(x)
print(x)
print(y)
z = hapkeModel.from_physic(y)
print(x)
print(y)
print(z)

70
4
[0.1 0.1 0.1 0.1]
(4,)
[0.1 0.1 0.1 0.1]
(4,)
[0.1 0.1 0.1 0.1]
[0.19 3.   0.1  0.1 ]
(4,)
=========== From physic ==========
[0.1 0.1 0.1 0.1]
[0.1 0.1 0.1 0.1]
[0.1 0.1 0.1 0.1]
[0.0513167  0.00333333 0.1        0.1       ]
[0.1 0.1 0.1 0.1]
[0.0513167  0.00333333 0.1        0.1       ]
[0.02599625 0.00011111 0.1        0.1       ]


## External model

In [10]:
externalPythonModel = lib.ExternalPythonModel("ShkuratovModel5p", "ShkuratovModel5pPython", "../../pytest/models/")

print(externalPythonModel.get_D_dimension())
print(externalPythonModel.get_L_dimension())

50
5


In [11]:
x = np.ones(externalPythonModel.get_L_dimension())

print(x)
print(x.shape)
print("=========== To physic ==========")
externalPythonModel.to_physic(x) # does nothing
print(x)
print(x.shape)
w = externalPythonModel.to_physic(x) # return vec size-like : w.shape() = (5,1)
print(x)
print(w)
print(w.shape)
print("=========== From physic ==========")
externalPythonModel.from_physic(x)
print(x)
y = externalPythonModel.from_physic(x)
print(x)
print(y)
z = externalPythonModel.from_physic(y)
print(x)
print(y)
print(z)

[1. 1. 1. 1. 1.]
(5,)
=========== To physic ==========
[1. 1. 1. 1. 1.]
(5,)
[1. 1. 1. 1. 1.]
[1.  1.5 1.7 1.5 1.5]
(5,)
=========== From physic ==========
[1. 1. 1. 1. 1.]
[1. 1. 1. 1. 1.]
[1.         0.66666667 0.53333333 0.66666667 0.66666667]
[1. 1. 1. 1. 1.]
[1.         0.66666667 0.53333333 0.66666667 0.66666667]
[1.         0.44444444 0.22222222 0.44444444 0.44444444]


In [12]:
with open('../../pytest/new_test_shkuratov.json', 'r') as f:
    data = json.load(f)

D = 50
row_size = D
col_size = 3

scalingCoeffs = [1.0,1.5,1.5,1.5,1.5]
offset = [0,0,0.2,0,0]



### TEST
N = 3
L = 5
variables = ["an", "mu1", "nu", "m", "mu2"]
photometries = np.empty((L,N))

# Read photometries
for l in range(L):
    for n in range(N):
        photometries[l,n] = (float(data[variables[l]][n]) - offset[l]) / scalingCoeffs[l]
        n+=1


# Read expected results
expected_results = np.empty((D,N))
n=0
for n in range(N):
    for d in range(D):
        expected_results[d,n] = float(data["y"][n][d])


# compute results from the model
# result = np.empty((D,))
assert_list = []
for n in range(N):
    result = externalPythonModel.F(photometries[:,n])
    assert_list.append(np.allclose(expected_results[:,n], result, rtol=1e-8))

print(result.shape)
print(assert_list)
print(False in assert_list)
print(True in assert_list)

print(expected_results[:10,n])
print(result[:10])

(50,)
[True, True, True]
False
True
[0.16476508 0.0988973  0.21693669 0.1171771  0.1004486  0.19843122
 0.13557557 0.30234728 0.14297123 0.20419017]
[0.16476508 0.0988973  0.21693669 0.1171771  0.1004486  0.19843122
 0.13557557 0.30234728 0.14297123 0.20419017]


## dataGeneration

In [17]:
model = lib.TestModel()
covariances = np.random.uniform(0, 0.0001, model.get_D_dimension())
stat_model = lib.GaussianStatModel("random", model, covariances, 12345)

RuntimeError: Caught an unknown exception!

In [14]:
x_gen, y_gen = stat_model.gen_data(10)
print(x_gen)
print(y_gen)

[[0.35762972 0.00393225 0.00388677 0.93426213]
 [0.40044262 0.01302971 0.94009241 0.9856864 ]
 [0.68938332 0.42042242 0.6912759  0.43128775]
 [0.55973557 0.61619145 0.91866556 0.12801226]
 [0.57445129 0.89488361 0.41973236 0.87014702]
 [0.20769053 0.41063606 0.80409551 0.77154433]
 [0.0286627  0.26076873 0.59863092 0.70017779]
 [0.68892448 0.02672894 0.00344757 0.644223  ]
 [0.46934336 0.1079424  0.48831784 0.63574481]
 [0.2071526  0.36668398 0.88220464 0.99782252]]
[[ 5.667968    0.25096795  1.33827538  3.81817162  0.14296009 -0.25096747
  -1.48135691  1.96152935 -0.89083316]
 [ 5.2690531   0.25327669  1.08503027  4.01944998  0.14922852 -0.2532526
  -1.23422853  1.42385521 -0.93788405]
 [ 4.44616581  0.38065083  0.57876082  2.30883985  0.19925705 -0.38062761
  -0.77797307  0.16145878 -0.53875156]
 [ 5.31124599  0.46296349  1.00797311  1.70497087  0.17498939 -0.46307461
  -1.18299329  1.14074515 -0.39779955]
 [ 5.55505458  0.61176005  0.51304759  3.58082557  0.1776296  -0.61181666
  -0

In [15]:
print(x_gen.shape)
print(y_gen.shape)

(10, 4)
(10, 9)


In [16]:
for n in range(x_gen.shape[0]):
    result = model.F(x_gen[n,:])
    print(np.allclose(y_gen[n,:], result, rtol=1e-3))

True
True
True
True
True
True
True
True
True
True
